In [1]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
data = pd.read_parquet("data/full_dataset.parquet")

In [65]:
# Fill missing data with average reading across stations at given time
variables = ['RH', 'DR', 'SQ', 'Q', 'temp', 'TD', 'FF', 'FH', 'P', 'N', 'VV','DD', 
             'rh', 'T10N', 'FX', 'WW', 'IX', 'M', 'R', 'S', 'O', 'Y', 'BC', 'CO', 
             'H2S', 'NH3', 'NOx', 'O3', 'Ox', 'SO2', 'no2', 'no2.1', 'pm10', 'pm25']

for var in variables:
    data[var] = data.groupby("time")[var].transform(lambda x: x.fillna(x.mean()))

# Check for remaining missing values in features
print(data[variables].isna().sum())

RH            0
DR            0
SQ            0
Q             0
temp          0
TD            0
FF            0
FH            0
P             0
N             0
VV            0
DD            0
rh            0
T10N          0
FX            0
WW         4384
IX       192676
M             0
R             0
S             0
O             0
Y             0
BC       424924
CO       222870
H2S      356967
NH3      410222
NOx      218118
O3       222981
Ox       222981
SO2      212124
no2      218118
no2.1    218118
pm10     329910
pm25     489691
dtype: int64


In [68]:
# Fill in remaining missing values with mean of each feature
data = data[variables].fillna(data[variables].mean())

In [69]:
# Handle class imbalance by reducing no of "Light-Moderate" samples
data["precipitation_category"] = pd.cut(data["RH"], bins=[-np.inf, 50, 100, np.inf], labels=[0,1,2])

class_labels = {0: "Light-Moderate", 1: "Heavy", 2: "Extreme"}

light_mod = data[data["precipitation_category"] == 0]
other_classes = data[data["precipitation_category"] != 0]

light_mod_downsampled = light_mod.sample(frac=0.00018, random_state=42)

data_balanced = pd.concat([light_mod_downsampled, other_classes])

In [70]:
# Precipitation category counts
category_counts = data_balanced["precipitation_category"].value_counts()
category_counts.index = category_counts.index.map(class_labels)
print(category_counts)

precipitation_category
Light-Moderate    128
Heavy             128
Extreme             6
Name: count, dtype: int64


In [71]:
# Prepare variables for modeling
# Variables division (excluding ZWR due to 0 records)
target = 'precipitation_category'
meteorological_features = ['RH', 'DR', 'SQ', 'Q', 'temp', 'TD', 'FF', 'FH', 'P', 'N', 'VV','DD', 
             'rh', 'T10N', 'FX', 'WW', 'IX', 'M', 'R', 'S', 'O', 'Y']
all_features = ['RH', 'DR', 'SQ', 'Q', 'temp', 'TD', 'FF', 'FH', 'P', 'N', 'VV','DD', 
             'rh', 'T10N', 'FX', 'WW', 'IX', 'M', 'R', 'S', 'O', 'Y', 'BC', 'CO', 
             'H2S', 'NH3', 'NOx', 'O3', 'Ox', 'SO2', 'no2', 'no2.1', 'pm10', 'pm25']

# Train-test split, small test set due to low amount of data after handling class imbalance
all_train_x, all_test_x, all_train_y, all_test_y = train_test_split(data_balanced[all_features], data_balanced[target], test_size=0.1, random_state=42)
train_x, test_x, train_y, test_y = train_test_split(data_balanced[meteorological_features], data_balanced[target], test_size=0.1, random_state=42)

### XGBoost

#### Meteorological data only

In [72]:
# Train XGBoost model
xgboost_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgboost_model.fit(train_x, train_y)

# Predict and evaluate XGBoost
xgboost_predictions = xgboost_model.predict(test_x) 

print("XGBoost Classification Report:")
print(classification_report(test_y, xgboost_predictions))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00         2

    accuracy                           1.00        27
   macro avg       1.00      1.00      1.00        27
weighted avg       1.00      1.00      1.00        27



c:\Users\majak\Desktop\Uni\Thesis\Extreme-precipitation\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:04:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


#### Including air quality data

In [73]:
# Train XGBoost model
all_xgboost_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
all_xgboost_model.fit(all_train_x, all_train_y)

# Predict and evaluate XGBoost
all_xgboost_predictions = all_xgboost_model.predict(all_test_x) 

print("XGBoost Classification Report (including all features):")
print(classification_report(all_test_y, all_xgboost_predictions))

c:\Users\majak\Desktop\Uni\Thesis\Extreme-precipitation\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:04:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Classification Report (including all features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00         2

    accuracy                           1.00        27
   macro avg       1.00      1.00      1.00        27
weighted avg       1.00      1.00      1.00        27



### SVM

#### Meteorological data only

In [74]:
# Train SVM model   
svm_model = SVC(kernel='rbf', class_weight='balanced')
svm_model.fit(train_x, train_y) 

# Predict and evaluate SVM
svm_predictions = svm_model.predict(test_x)

print("SVM Classification Report:")
print(classification_report(test_y, svm_predictions))

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        11
           1       0.36      0.29      0.32        14
           2       0.00      0.00      0.00         2

    accuracy                           0.15        27
   macro avg       0.12      0.10      0.11        27
weighted avg       0.19      0.15      0.17        27



In [75]:
# Check for remaining missing values in features
print(data_balanced[variables].isna().sum())

RH       0
DR       0
SQ       0
Q        0
temp     0
TD       0
FF       0
FH       0
P        0
N        0
VV       0
DD       0
rh       0
T10N     0
FX       0
WW       0
IX       0
M        0
R        0
S        0
O        0
Y        0
BC       0
CO       0
H2S      0
NH3      0
NOx      0
O3       0
Ox       0
SO2      0
no2      0
no2.1    0
pm10     0
pm25     0
dtype: int64


#### Including air quality data

In [76]:
# Train SVM model with all features
all_svm_model = SVC(kernel='rbf', class_weight='balanced')
all_svm_model.fit(all_train_x, all_train_y)

# Predict and evaluate SVM
all_svm_predictions = all_svm_model.predict(all_test_x)

print("SVM Classification Report (including all features):")
print(classification_report(all_test_y, all_svm_predictions))

SVM Classification Report (including all features):
              precision    recall  f1-score   support

           0       0.33      0.09      0.14        11
           1       0.40      0.29      0.33        14
           2       0.00      0.00      0.00         2

    accuracy                           0.19        27
   macro avg       0.24      0.13      0.16        27
weighted avg       0.34      0.19      0.23        27

